In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

data = pd.read_excel("skincare-labeled.xlsx")

# Data sudah diberi label sentimen secara manual (0=Positive, 1=Neutral, 2=Negative)
data.head()

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_nb = 1
fold_svm = 1

nb_results = []

svm_results = []

label_map = {0: "Positive", 1: "Neutral", 2: "Negative"}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

In [ ]:
# TF-IDF dan Pipeline untuk Naive Bayes
tfidf_nb = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
nb_pipeline = Pipeline([
    ('tfidf', tfidf_nb),
    ('clf', MultinomialNB())
])

nb_param_grid = {
    'clf__alpha': [0.1, 0.5, 1.0, 1.5, 2.0]
}

In [ ]:
# Iterasi untuk setiap fold
for train_idx, eval_idx in skf.split(data["normalized_text"], data["label"]):
    print(f"\n===== Naive Bayes FOLD {fold_nb} =====")

    # Bagi data menjadi data latih (train) dan data evaluasi (eval)
    X_train, y_train = data["normalized_text"].iloc[train_idx], data["label"].iloc[train_idx]
    X_eval, y_eval = data["normalized_text"].iloc[eval_idx], data["label"].iloc[eval_idx]

    # Menghitung bobot manual
    classes_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    weight_dict = dict(zip(np.unique(y_train), classes_weights))
    sample_weights = y_train.map(weight_dict).values

    # Grid Search & Training
    nb_grid = GridSearchCV(nb_pipeline, nb_param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
    nb_grid.fit(X_train, y_train, clf__sample_weight=sample_weights)

    print(f"Parameter Naive Bayes Terbaik: {nb_grid.best_params_}")

    # Prediksi
    y_pred_nb = nb_grid.best_estimator_.predict(X_eval)

    # Hitung metrik evaluasi
    acc = accuracy_score(y_eval, y_pred_nb)
    precision, recall, f1, _ = precision_recall_fscore_support(y_eval, y_pred_nb, average='macro', zero_division=0)

    nb_results.append({
        "fold": fold_nb,
        "best_params": nb_grid.best_params_,
        "accuracy_nb": acc,
        "precision_macro_nb": precision,
        "recall_macro_nb": recall,
        "f1_macro_nb": f1
    })

    # Hitung distribusi sentimen
    sentiment_dist = pd.Series(y_pred_nb).value_counts().sort_index()
    sentiment_dist.index = sentiment_dist.index.map(label_map)
    sentiment_summary = pd.DataFrame({
        "Jumlah": sentiment_dist,
        "Persentase (%)": (sentiment_dist / sentiment_dist.sum() * 100).round(2)
    })

    print(f"\nSentiment Distribution Naive Bayes Fold {fold_nb}")
    print(sentiment_summary)

    # Pie Chart Distribusi Sentimen
    sentiment_summary["Jumlah"].plot(kind="pie", autopct="%1.1f%%", figsize=(3,3), title=f"Distribusi Sentimen Naive Bayes Fold {fold_nb}")
    plt.ylabel("")
    plt.show()

    # Classification Report
    print(f"\nClassification Report Naive Bayes Fold {fold_nb}")
    print(classification_report(y_eval, y_pred_nb, target_names=["Positive", "Neutral", "Negative"]))

    # Lanjut ke fold berikutnya
    fold_nb += 1

In [ ]:
# Mengambil nilai F1-Score Macro dari seluruh fold
f1_nb = [result["f1_macro_nb"] for result in nb_results]

print("F1-Score Macro Naive Bayes per Fold:", f1_nb)

In [ ]:
avg_accuracy_nb = np.mean([result["accuracy_nb"] for result in nb_results])
avg_precision_nb = np.mean([result["precision_macro_nb"] for result in nb_results])
avg_recall_nb = np.mean([result["recall_macro_nb"] for result in nb_results])
avg_f1score_nb = np.mean([result["f1_macro_nb"] for result in nb_results])

print(f"Average Naive Bayes Accuracy : {avg_accuracy_nb:.4f}")
print(f"Average Naive Bayes Precision: {avg_precision_nb:.4f}")
print(f"Average Naive Bayes Recall   : {avg_recall_nb:.4f}")
print(f"Average Naive Bayes F1-Score : {avg_f1score_nb:.4f}")

**Step 5: SVM (Kernel Linear)**

In [ ]:
# TF-IDF dan Pipeline untuk SVM
tfidf_svm = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
svm_pipeline = Pipeline([
    ('tfidf', tfidf_svm),
    # Tetapkan kernel='linear' langsung di dalam SVC agar menjadi parameter tetap
    ('clf', SVC(kernel='linear', class_weight='balanced', random_state=42))
])

svm_param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100]
}

In [ ]:
# Iterasi untuk setiap fold
for train_idx, eval_idx in skf.split(data["normalized_text"], data["label"]):
    print(f"\n===== SVM FOLD {fold_svm} =====")

    # Bagi data menjadi data latih (train) dan data evaluasi (eval)
    X_train, y_train = data["normalized_text"].iloc[train_idx], data["label"].iloc[train_idx]
    X_eval, y_eval = data["normalized_text"].iloc[eval_idx], data["label"].iloc[eval_idx]

    # Grid Search & Training SVM
    svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
    svm_grid.fit(X_train, y_train)

    print(f"Parameter SVM Terbaik: {svm_grid.best_params_}")

    # Prediksi
    y_pred_svm = svm_grid.best_estimator_.predict(X_eval)

    # Hitung metrik evaluasi
    acc = accuracy_score(y_eval, y_pred_svm)
    prec, rec, f1, _ = precision_recall_fscore_support(y_eval, y_pred_svm, average='macro', zero_division=0)

    svm_results.append({
        "fold": fold_svm,
        "best_params": svm_grid.best_params_,
        "accuracy_svm": acc,
        "precision_macro_svm": prec,
        "recall_macro_svm": rec,
        "f1_macro_svm": f1
    })

    # Hitung distribusi sentimen
    sentiment_dist = pd.Series(y_pred_svm).value_counts().sort_index()
    sentiment_dist.index = sentiment_dist.index.map(label_map)
    sentiment_summary = pd.DataFrame({
        "Jumlah": sentiment_dist,
        "Persentase (%)": (sentiment_dist / sentiment_dist.sum() * 100).round(2)
    })

    print(f"\nSentiment Distribution SVM Fold {fold_svm}")
    print(sentiment_summary)

    # Pie Chart Distribusi Sentimen
    sentiment_summary["Jumlah"].plot(kind="pie", autopct="%1.1f%%", figsize=(3,3), title=f"Distribusi Sentimen SVM Fold {fold_svm}")
    plt.ylabel("")
    plt.show()

    # Classification Report
    print(f"\nClassification Report SVM Fold {fold_svm}")
    print(classification_report(y_eval, y_pred_svm, target_names=["Positive", "Neutral", "Negative"]))

    # Lanjut ke fold berikutnya
    fold_svm += 1

In [ ]:
f1_svm = [result["f1_macro_svm"] for result in svm_results]

print("F1-Score Macro SVM per Fold:", f1_svm)

In [ ]:
avg_accuracy_svm = np.mean([result["accuracy_svm"] for result in svm_results])
avg_precision_svm = np.mean([result["precision_macro_svm"] for result in svm_results])
avg_recall_svm = np.mean([result["recall_macro_svm"] for result in svm_results])
avg_f1score_svm = np.mean([result["f1_macro_svm"] for result in svm_results])

print(f"Average SVM Accuracy: {avg_accuracy_svm:.4f}")
print(f"Average SVM Precision: {avg_precision_svm:.4f}")
print(f"Average SVM Recall: {avg_recall_svm:.4f}")
print(f"Average SVM F1-Score: {avg_f1score_svm:.4f}")